TD Final noté : Analyse des Avis et Alertes ANSSI avec
Enrichissement des CVE

● Extraire les données des flux RSS des avis et alertes ANSSI.

● Identifier les CVE mentionnées dans les bulletins.

● Enrichir les CVE avec des informations complémentaires via des API externes.


● Consolider les données dans un format exploitable (DataFrame pandas).


● Analyser et visualiser le DataFrame obtenu (vulnérabilités critiques, scores...)


● Modèles Machine learning


● Générer des alertes personnalisées pour les produits affectés et envoyer des notifications par email.

## Étape 1 — Extraction des flux RSS ANSSI

Les avis et alertes du CERT-FR sont récupérés via la fonction réutilisable
`anssi.feeds.fetch_all_feeds`. Chaque entrée est normalisée en un objet
`BulletinEntry` : **identifiant** (`CERTFR-...`), **type** (Avis / Alerte),
**titre**, **description**, **date de publication** (UTC) et **lien**.
La fonction est robuste aux champs manquants, normalise l'encodage et les
dates, et dédoublonne les bulletins.

Deux sources produisent le **même** schéma :

- **`source="rss"`** — le flux RSS live du CERT-FR ; ne renvoie que les
  bulletins les plus récents. C'est l'entrée décrite par le sujet.
- **`source="local"`** — les JSON pré-téléchargés (`data/data/`), qui couvrent
  **l'historique complet** (≈ 4 100 bulletins depuis 2021). C'est la base de
  travail des étapes d'analyse et de ML.

> **Accès responsable** (§8 du sujet) : en mode RSS, le flux brut est mis en
> cache (`data/feeds_cache/`) et relu au lieu d'être re-téléchargé, avec un
> délai de 2 s entre deux requêtes réseau.

In [ ]:
from anssi.feeds import fetch_all_feeds, feed_to_dataframe

# Flux RSS live (derniers bulletins) — la source décrite par le sujet.
recents = fetch_all_feeds(source="rss")
print(f"Flux RSS  : {len(recents)} bulletins récents")

# Corpus complet (JSON pré-téléchargés) — base de travail pour l'analyse.
bulletins = fetch_all_feeds(source="local")
print(f"Corpus local : {len(bulletins)} bulletins (avis + alertes)")

df_bulletins = feed_to_dataframe(bulletins)
df_bulletins.head()

## Étape 2 — Extraction des CVE

Chaque bulletin peut référencer des CVE dans plusieurs endroits :

| Source | Champ(s) | Fiabilité |
|---|---|---|
| Structurée | `cves[].name` | ✅ Haute — liste dédiée |
| Texte libre | `summary`, `content` | ✅ Bonne — mentions directes |
| Révisions | `revisions[].description` | ✅ Bonne — CVE ajoutés/retirés lors de mises à jour |
| Advisories tiers | `vendor_advisories[].title`, `.url` | ⚠️ Moyenne — CVE dans des URLs (risque de concaténation) |
| Systèmes affectés | `affected_systems[].description` | ⚠️ Rare |

**Validation** : les CVE sont filtrés par année (1999 ≤ year ≤ now+1) pour éliminer les faux positifs évidents (ex. `CVE-2203-23955`). Les identifiants douteux issus d'URLs (ex. `CVE-2023-2140712`) seront confirmés ou rejetés lors de l'enrichissement MITRE (étape 3).

In [ ]:
from anssi.cves import extract_all_cves, cves_to_dataframe

bulletin_cves = extract_all_cves()

total_cves = len({cve for b in bulletin_cves for cve in b.cves})
with_cves  = sum(1 for b in bulletin_cves if b.cves)
with_extra = sum(1 for b in bulletin_cves if b.cves_extra)

print(f"Bulletins traités         : {len(bulletin_cves)}")
print(f"Avec au moins 1 CVE       : {with_cves}")
print(f"Sans CVE                  : {len(bulletin_cves) - with_cves}")
print(f"Avec CVE hors clé 'cves'  : {with_extra}  (trouvés dans summary/revisions/vendor_advisories)")
print(f"CVE distincts (corpus)    : {total_cves:,}")

# Format long : une ligne par (bulletin, CVE) — base pour l'étape 4
df_cves = cves_to_dataframe(bulletin_cves)
print(f"\nDataFrame long : {df_cves.shape[0]:,} lignes × {df_cves.shape[1]} colonnes")
df_cves[df_cves["cve"].notna()].head()

## Étape 3 — Enrichissement des CVE (MITRE + FIRST/EPSS)

Chaque CVE est enrichi via deux sources :

| Source | Données | Particularités gérées |
|---|---|---|
| **MITRE** | Description, CVSS, CWE, produits/versions | 54 % sans CVSS ; priorité `v4.0 > v3.1 > v3.0 > v2.0` ; `type="cwe"` (minuscule) ; `metrics: null` ; versions avec `lessThan`/`lessThanOrEqual` |
| **FIRST** | Score EPSS, percentile | Valeurs manquantes → `None` |

Toutes les valeurs absentes sont `None` (jamais NaN) pour compatibilité pandas.

In [ ]:
from anssi.enrichment import enrich_cves, enrichment_to_dataframe

# Récupère tous les CVE uniques du corpus (étape 2)
all_cve_ids = list({cve for b in bulletin_cves for cve in b.cves})
print(f"CVE distincts à enrichir : {len(all_cve_ids):,}")

# Enrichissement depuis les données locales (§8 du sujet)
enrichments = enrich_cves(all_cve_ids, source="local")

mitre_ok = sum(1 for e in enrichments.values() if e.mitre_available)
first_ok = sum(1 for e in enrichments.values() if e.first_available)
with_cvss = sum(1 for e in enrichments.values() if e.mitre.cvss_score is not None)
with_cwe  = sum(1 for e in enrichments.values() if e.mitre.cwe_id is not None)
with_epss = sum(1 for e in enrichments.values() if e.first.epss_score is not None)

print(f"Données MITRE disponibles : {mitre_ok:,} / {len(enrichments):,}")
print(f"Données FIRST disponibles : {first_ok:,} / {len(enrichments):,}")
print(f"Avec score CVSS           : {with_cvss:,}  ({100*with_cvss/len(enrichments):.1f}%)")
print(f"Avec CWE                  : {with_cwe:,}  ({100*with_cwe/len(enrichments):.1f}%)")
print(f"Avec score EPSS           : {with_epss:,}  ({100*with_epss/len(enrichments):.1f}%)")

df_enrichment = enrichment_to_dataframe(enrichments)
df_enrichment.head()

## Étape 4 — Consolidation du DataFrame

Toutes les données (bulletins ANSSI, CVE extraites, enrichissement MITRE + EPSS)
sont assemblées dans un seul DataFrame à la granularité **(bulletin\_id, cve)**.

| Origine | Colonnes |
|---|---|
| Bulletin | bulletin\_id, bulletin\_type, title, date\_publication, year\_publication |
| CVE | cve, cve\_year |
| MITRE | cve\_description, cvss\_version, cvss\_score, cvss\_severity, cvss\_vector, cwe\_id, cwe\_description, vendors, products, versions |
| FIRST | epss\_score, epss\_percentile, epss\_date |
| Dérivé | is\_critical (score ≥ 9.0 ou severity == CRITICAL) |

Les bulletins sans CVE sont conservés (une ligne avec ).
Les valeurs manquantes sont  (jamais NaN), les types sont typés
(, , , ) pour une utilisation directe en ML.

In [ ]:
from anssi.consolidation import consolidate

df = consolidate(source="local")

print(f"DataFrame consolidé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Avec CVE            : {df["cve"].notna().sum():,} lignes")
print(f"Sans CVE            : {df["cve"].isna().sum():,} lignes")
print(f"CVE critiques       : {df["is_critical"].sum():,}")
print(f"Couverture MITRE    : {df["mitre_available"].sum():,} / {df["cve"].notna().sum():,}")
print(f"Couverture EPSS     : {df["first_available"].sum():,} / {df["cve"].notna().sum():,}")
print()
print(df.dtypes.to_string())
df.head()

## Étape 5 — Analyse exploratoire et visualisation

On charge le CSV produit à l'étape 4 et on explore les données enrichies :
distribution des scores CVSS/EPSS, types CWE, éditeurs les plus touchés,
évolution temporelle, corrélations...


### Chargement et préparation des données


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style global
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE = sns.color_palette('muted')
SEV_COLORS = {'LOW': '#4caf50', 'MEDIUM': '#ff9800', 'HIGH': '#f44336', 'CRITICAL': '#9c27b0', 'NONE': '#9e9e9e'}

df = pd.read_csv('data/consolidated.csv', low_memory=False)
df['date_publication'] = pd.to_datetime(df['date_publication'], utc=True, errors='coerce')
df['year_month'] = df['date_publication'].dt.to_period('M')

# CVE uniques (pour analyses sans doublon bulletin)
df_cve = df.dropna(subset=['cve']).drop_duplicates(subset=['cve'])

# Vendeurs : une ligne par (bulletin_id, cve, vendor)
df_vendors = (
    df.dropna(subset=['cve', 'vendors'])
    .drop_duplicates(subset=['cve'])
    .assign(vendor=lambda x: x['vendors'].str.split(';'))
    .explode('vendor')
    .assign(vendor=lambda x: x['vendor'].str.strip())
    .query('vendor != ""')
)

print(f"Corpus total           : {len(df):,} lignes")
print(f"CVE distincts          : {df['cve'].nunique():,}")
print(f"Bulletins distincts    : {df['bulletin_id'].nunique():,}")
print(f"Couverture CVSS        : {df_cve['cvss_score'].notna().sum():,} / {len(df_cve):,} CVE ({100*df_cve['cvss_score'].notna().mean():.1f}%)")
print(f"Couverture EPSS        : {df_cve['epss_score'].notna().sum():,} / {len(df_cve):,} CVE ({100*df_cve['epss_score'].notna().mean():.1f}%)")
print(f"Couverture CWE         : {df_cve['cwe_id'].notna().sum():,} / {len(df_cve):,} CVE ({100*df_cve['cwe_id'].notna().mean():.1f}%)")
print(f"Période couverte       : {df['date_publication'].min().date()} -> {df['date_publication'].max().date()}")


Le corpus couvre plus de **5 ans** de bulletins ANSSI (2021–2026), soit 37 287 CVE distincts.
Environ **78 %** des CVE ont un score CVSS disponible dans les données MITRE locales.
Le taux identique pour EPSS confirme que les deux sources sont corrélées (même population couverte).


### 1. Répartition des niveaux de sévérité CVSS


In [ ]:
sev_order = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
sev_counts = (df_cve['cvss_severity']
              .dropna()
              .pipe(lambda s: s[s.isin(sev_order)])
              .value_counts()
              .reindex(sev_order))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Barplot
colors = [SEV_COLORS[s] for s in sev_order]
axes[0].bar(sev_counts.index, sev_counts.values, color=colors, edgecolor='white', linewidth=0.8)
for i, (label, val) in enumerate(zip(sev_counts.index, sev_counts.values)):
    axes[0].text(i, val + 80, f'{val:,}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Nombre de CVE par niveau de sévérité', fontweight='bold')
axes[0].set_xlabel('Sévérité CVSS')
axes[0].set_ylabel('Nombre de CVE uniques')

# Pie
axes[1].pie(sev_counts.values, labels=sev_counts.index, colors=colors,
            autopct='%1.1f%%', startangle=140, pctdistance=0.75,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('Distribution en pourcentage', fontweight='bold')

plt.suptitle('Sévérité CVSS — corpus ANSSI 2021–2026', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


Les vulnérabilités de niveau **HIGH** et **MEDIUM** dominent (~44 % et ~45 %),
tandis que les **CRITICAL** représentent environ **5 %** — soit ~1 500 CVE à traiter
en priorité absolue. Les LOW sont rares (~8 %), ce qui reflète la politique de
l'ANSSI qui ne publie généralement des bulletins que pour des menaces significatives.


### 2. Distribution des scores CVSS


In [ ]:
cvss = df_cve['cvss_score'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogramme
axes[0].hist(cvss, bins=40, color=PALETTE[0], edgecolor='white', linewidth=0.5)
axes[0].axvline(cvss.median(), color='crimson', linestyle='--', label=f'Médiane : {cvss.median():.1f}')
axes[0].axvline(cvss.mean(), color='orange', linestyle='--', label=f'Moyenne : {cvss.mean():.1f}')
axes[0].set_title('Distribution des scores CVSS', fontweight='bold')
axes[0].set_xlabel('Score CVSS')
axes[0].set_ylabel('Nombre de CVE')
axes[0].legend()

# Boxplot par version CVSS
cvss_ver = df_cve[df_cve['cvss_score'].notna()][['cvss_version', 'cvss_score']].copy()
cvss_ver['cvss_version'] = cvss_ver['cvss_version'].astype(str)
order = sorted(cvss_ver['cvss_version'].dropna().unique())
sns.boxplot(data=cvss_ver, x='cvss_version', y='cvss_score', order=order,
            palette='muted', ax=axes[1])
axes[1].set_title('Score CVSS selon la version du standard', fontweight='bold')
axes[1].set_xlabel('Version CVSS')
axes[1].set_ylabel('Score CVSS')

plt.suptitle('Analyse des scores CVSS', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


La distribution présente deux modes distincts : un pic autour de **7-8** (HIGH) et un
second autour de **5-6** (MEDIUM). Le score médian (~7) indique que la moitié des
vulnérabilités signalées par l'ANSSI sont de sévérité élevée ou critique.
Les CVSSv2 tendent à produire des scores légèrement plus bas que CVSSv3 pour
des vulnérabilités similaires — effet bien documenté du changement de méthodologie.


### 3. Distribution des scores EPSS


In [ ]:
epss = df_cve['epss_score'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogramme échelle log
axes[0].hist(epss, bins=60, color=PALETTE[2], edgecolor='white', linewidth=0.4)
axes[0].axvline(epss.median(), color='crimson', linestyle='--', label=f'Médiane : {epss.median():.4f}')
axes[0].set_title('Distribution des scores EPSS (échelle log)', fontweight='bold')
axes[0].set_xlabel('Score EPSS')
axes[0].set_ylabel('Nombre de CVE')
axes[0].set_yscale('log')
axes[0].legend()

# Courbe cumulative
axes[1].plot(np.sort(epss), np.linspace(0, 1, len(epss)), color=PALETTE[2], linewidth=2)
axes[1].axhline(0.9, color='crimson', linestyle=':', alpha=0.7, label='90e percentile')
p90 = epss.quantile(0.9)
axes[1].axvline(p90, color='orange', linestyle=':', alpha=0.7, label=f'EPSS @P90 = {p90:.3f}')
axes[1].set_title('Distribution cumulative EPSS', fontweight='bold')
axes[1].set_xlabel('Score EPSS')
axes[1].set_ylabel('Fraction de CVE')
axes[1].legend()

plt.suptitle('Analyse des scores EPSS', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"90% des CVE ont un EPSS < {p90:.4f}")
print(f"CVE avec EPSS > 0.5 (très probable exploitation) : {(epss > 0.5).sum():,} ({100*(epss>0.5).mean():.1f}%)")


La distribution EPSS est fortement asymétrique : **90 % des CVE ont une probabilité
d'exploitation inférieure à 5 %**. Seule une minorité dépasse 0.5 — ce sont les cibles
prioritaires pour les équipes de réponse.
La courbe cumulative confirme la loi de Pareto : une petite fraction des CVE concentre
l'essentiel du risque réel d'exploitation.


### 4. Corrélation CVSS / EPSS


In [ ]:
df_both = df_cve[df_cve['cvss_score'].notna() & df_cve['epss_score'].notna()].copy()
df_both['cvss_severity'] = df_both['cvss_severity'].fillna('NONE')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter CVSS vs EPSS coloré par sévérité
for sev in ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']:
    sub = df_both[df_both['cvss_severity'] == sev]
    axes[0].scatter(sub['cvss_score'], sub['epss_score'],
                    c=SEV_COLORS[sev], label=sev, alpha=0.25, s=8, rasterized=True)
axes[0].set_title('Score CVSS vs EPSS par sévérité', fontweight='bold')
axes[0].set_xlabel('Score CVSS')
axes[0].set_ylabel('Score EPSS')
axes[0].legend(markerscale=3)

# Heatmap de corrélation
num_cols = ['cvss_score', 'epss_score', 'epss_percentile', 'cve_year']
corr = df_cve[num_cols].dropna().corr()
labels = ['CVSS Score', 'EPSS Score', 'EPSS Percentile', 'Année CVE']
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            xticklabels=labels, yticklabels=labels,
            ax=axes[1], linewidths=0.5, vmin=-1, vmax=1)
axes[1].set_title('Heatmap de corrélation', fontweight='bold')

plt.suptitle('Relation entre score CVSS et probabilité d\'exploitation (EPSS)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

r = df_both['cvss_score'].corr(df_both['epss_score'])
print(f"Corrélation de Pearson CVSS / EPSS : r = {r:.3f}")


La corrélation CVSS/EPSS est **faible** (~0.1–0.2), ce qui est contre-intuitif mais bien
connu dans la littérature : un score CVSS élevé ne prédit pas l'exploitation réelle.
Certaines vulnérabilités MEDIUM sont très exploitées (EPSS élevé) pendant que des
CRITICAL restent théoriques. **CVSS mesure la sévérité potentielle, EPSS mesure
le risque réel** — les deux dimensions sont complémentaires, pas redondantes.


### 5. Types de vulnérabilités (CWE) les plus fréquents


In [ ]:
# Fusion avec descriptions
cwe_df = (df_cve[df_cve['cwe_id'].notna()][['cwe_id', 'cwe_description']]
          .copy()
          .assign(cwe_id=lambda x: x['cwe_id'].str.strip()))
cwe_counts = cwe_df['cwe_id'].value_counts().head(15)

# Label court : CWE-XXX + début de description
desc_map = (cwe_df.drop_duplicates('cwe_id')
            .set_index('cwe_id')['cwe_description']
            .str[:40].to_dict())
labels = [f"{cwe}  –  {desc_map.get(cwe, '')[:35]}" for cwe in cwe_counts.index]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(labels[::-1], cwe_counts.values[::-1],
               color=sns.color_palette('Blues_r', len(cwe_counts)), edgecolor='white')
for bar, val in zip(bars, cwe_counts.values[::-1]):
    ax.text(bar.get_width() + 15, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
ax.set_title('Top 15 des types de vulnérabilités (CWE) — CVE uniques', fontweight='bold')
ax.set_xlabel('Nombre de CVE')
ax.set_xlim(0, cwe_counts.max() * 1.15)
plt.tight_layout()
plt.show()


**CWE-416** (Use-After-Free) domine largement — c'est une vulnérabilité mémoire
très répandue dans le code C/C++ (noyau Linux, navigateurs). **CWE-20** (validation
d'entrée insuffisante) et **CWE-79** (XSS) confirment les faiblesses classiques.
La prédominance des bugs mémoire (CWE-416, 125, 122, 787) illustre la fragilité
des systèmes écrits en langages non memory-safe.


### 6. Éditeurs les plus touchés


In [ ]:
top_vendors = df_vendors['vendor'].value_counts().head(12)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot nombre de CVE
colors_v = sns.color_palette('Oranges_r', len(top_vendors))
axes[0].barh(top_vendors.index[::-1], top_vendors.values[::-1],
             color=colors_v[::-1], edgecolor='white')
for i, val in enumerate(top_vendors.values[::-1]):
    axes[0].text(val + 200, i, f'{val:,}', va='center', fontsize=9)
axes[0].set_title('Top 12 éditeurs — nombre de CVE uniques affectés', fontweight='bold')
axes[0].set_xlabel('Nombre de CVE')
axes[0].set_xlim(0, top_vendors.max() * 1.15)

# Boxplot CVSS par éditeur
top8 = top_vendors.head(8).index.tolist()
df_v8 = df_vendors[df_vendors['vendor'].isin(top8) & df_vendors['cvss_score'].notna()]
order8 = df_v8.groupby('vendor')['cvss_score'].median().sort_values(ascending=False).index
sns.boxplot(data=df_v8, y='vendor', x='cvss_score', order=order8,
            palette='Oranges', ax=axes[1])
axes[1].set_title('Distribution CVSS pour les 8 éditeurs principaux', fontweight='bold')
axes[1].set_xlabel('Score CVSS')
axes[1].set_ylabel('')

plt.suptitle('Analyse par éditeur', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Linux** domine massivement en volume (noyau Linux + distributions), suivi de
**Microsoft** et **Red Hat**. Le boxplot révèle que Microsoft et Siemens présentent
des médianes CVSS plus élevées que Linux, dont les vulnérabilités sont souvent
détectées tôt et scorées modérément. Siemens (systèmes industriels / OT) affiche
les scores les plus dispersés — caractéristique des environnements ICS/SCADA.


### 7. Évolution temporelle des vulnérabilités


In [ ]:
# CVE uniques par mois (date du bulletin)
monthly = (df.dropna(subset=['cve'])
           .drop_duplicates(subset=['cve'])
           .assign(month=lambda x: x['date_publication'].dt.to_period('M'))
           .groupby('month', observed=True)
           .agg(total=('cve', 'count'),
                critical=('is_critical', 'sum'))
           .reset_index()
           .assign(month=lambda x: x['month'].dt.to_timestamp()))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].fill_between(monthly['month'], monthly['total'], alpha=0.4, color=PALETTE[0])
axes[0].plot(monthly['month'], monthly['total'], color=PALETTE[0], linewidth=1.5)
axes[0].set_title('Nombre de CVE distincts publiés par mois', fontweight='bold')
axes[0].set_ylabel('CVE / mois')

axes[1].fill_between(monthly['month'], monthly['critical'], alpha=0.5, color='crimson')
axes[1].plot(monthly['month'], monthly['critical'], color='crimson', linewidth=1.5)
axes[1].set_title('Dont CVE critiques (CVSS ≥ 9.0 ou sévérité CRITICAL)', fontweight='bold')
axes[1].set_ylabel('CVE critiques / mois')
axes[1].set_xlabel('Date')

plt.suptitle('Évolution temporelle des vulnérabilités ANSSI', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


On observe une **accélération nette depuis 2022**, avec des pics notables lors de
crises majeures (ex. Log4Shell fin 2021, vulnérabilités Windows massives 2023–2024).
Les CVE critiques suivent la même tendance mais restent proportionnellement stables,
suggérant que la croissance est portée par une meilleure détection, pas uniquement
par une aggravation de la menace.


### 8. Avis vs Alertes — comparaison


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Nombre de bulletins par type
type_counts = df.drop_duplicates('bulletin_id')['bulletin_type'].value_counts()
axes[0].bar(type_counts.index, type_counts.values,
            color=[PALETTE[0], PALETTE[3]], edgecolor='white', linewidth=1)
for i, val in enumerate(type_counts.values):
    axes[0].text(i, val + 10, f'{val:,}', ha='center', fontweight='bold')
axes[0].set_title('Nombre de bulletins par type', fontweight='bold')
axes[0].set_ylabel('Bulletins')

# Distribution CVSS : Avis vs Alertes
for btype, color in [('Avis', PALETTE[0]), ('Alerte', PALETTE[3])]:
    sub = df_cve[(df_cve['bulletin_type'] == btype) & df_cve['cvss_score'].notna()]
    axes[1].hist(sub['cvss_score'], bins=30, alpha=0.6, label=btype,
                 color=color, edgecolor='white', linewidth=0.3, density=True)
axes[1].set_title('Distribution CVSS : Avis vs Alertes (densité)', fontweight='bold')
axes[1].set_xlabel('Score CVSS')
axes[1].set_ylabel('Densité')
axes[1].legend()

plt.suptitle('Avis vs Alertes ANSSI', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

for btype in ['Avis', 'Alerte']:
    sub = df_cve[(df_cve['bulletin_type'] == btype) & df_cve['cvss_score'].notna()]
    print(f"{btype:7} — médiane CVSS : {sub['cvss_score'].median():.1f}  |  "
          f"% critique : {100*(sub['cvss_score']>=9).mean():.1f}%")


Les **Alertes** représentent une infime fraction des bulletins (~2 %) mais couvrent
des vulnérabilités activement exploitées. Leur distribution CVSS est décalée vers
les scores hauts par rapport aux Avis — confirmant qu'une alerte ANSSI signifie
un danger immédiat, pas seulement potentiel.


### 9. Matrice de priorisation : CVSS × EPSS


In [ ]:
df_p = df_cve[df_cve['cvss_score'].notna() & df_cve['epss_score'].notna()].copy()

# Quadrants de risque
HIGH_CVSS, HIGH_EPSS = 7.0, 0.1
df_p['quadrant'] = 'Faible risque'
df_p.loc[(df_p['cvss_score'] >= HIGH_CVSS) & (df_p['epss_score'] < HIGH_EPSS), 'quadrant'] = 'Sévère mais rare'
df_p.loc[(df_p['cvss_score'] < HIGH_CVSS) & (df_p['epss_score'] >= HIGH_EPSS), 'quadrant'] = 'Exploité mais modéré'
df_p.loc[(df_p['cvss_score'] >= HIGH_CVSS) & (df_p['epss_score'] >= HIGH_EPSS), 'quadrant'] = 'PRIORITÉ ABSOLUE'

q_colors = {
    'PRIORITÉ ABSOLUE': '#9c27b0',
    'Sévère mais rare': '#f44336',
    'Exploité mais modéré': '#ff9800',
    'Faible risque': '#9e9e9e',
}

fig, ax = plt.subplots(figsize=(10, 7))
for quad, color in q_colors.items():
    sub = df_p[df_p['quadrant'] == quad]
    ax.scatter(sub['cvss_score'], sub['epss_score'],
               c=color, label=f'{quad} ({len(sub):,})',
               alpha=0.3, s=10, rasterized=True)

ax.axvline(HIGH_CVSS, color='black', linestyle='--', alpha=0.4, linewidth=1)
ax.axhline(HIGH_EPSS, color='black', linestyle='--', alpha=0.4, linewidth=1)
ax.set_xlabel('Score CVSS (sévérité)', fontsize=12)
ax.set_ylabel('Score EPSS (probabilité d\'exploitation)', fontsize=12)
ax.set_title('Matrice de priorisation des CVE', fontweight='bold', fontsize=14)
ax.legend(markerscale=3, loc='upper left')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print('Répartition par quadrant :')
print(df_p['quadrant'].value_counts().to_string())


Le quadrant **PRIORITÉ ABSOLUE** (CVSS ≥ 7 ET EPSS ≥ 0.1) concentre les CVE
les plus dangereux : score de gravité élevé **et** probabilité réelle d'exploitation.
C'est ce sous-ensemble qui doit déclencher les alertes immédiates (étape 7).
La majorité des CVE sévères restent théoriques (quadrant rouge) — soulignant
l'intérêt d'**associer CVSS et EPSS** plutôt que de se fier à l'un ou l'autre.
